# Exploratory Data Analysis (EDA) — Masterclass

A reusable, comprehensive EDA reference covering:
- **Data Overview**: Shape, dtypes, info, describe
- **Missing Values**: Detection, visualization, imputation strategies
- **Univariate Analysis**: Distributions, histograms, box plots, value counts
- **Bivariate Analysis**: Scatter, correlation, pair plots
- **Multivariate Analysis**: Heatmaps, grouped analysis
- **Outlier Detection**: IQR, Z-score methods
- **Data Quality Checklist**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

# Configure plots
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load & First Look

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f'Shape: {df.shape}')
print(f'\nColumn Names: {list(df.columns)}')
print(f'\nTarget: MedHouseVal (Median House Value in $100K)')
df.head()

In [ ]:
# Data types and memory
print('='*60)
print('DATA INFO')
print('='*60)
df.info()
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

In [ ]:
# Statistical summary
print('='*60)
print('DESCRIPTIVE STATISTICS')
print('='*60)
df.describe().T.style.background_gradient(cmap='YlGnBu', axis=1)

## 2. Missing Values Analysis

In [ ]:
# Missing values summary
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
    'Data Type': df.dtypes
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing) == 0:
    print('No missing values found! Dataset is complete.')
else:
    print('Missing Values:')
    display(missing)

# Demonstrate missing value handling patterns (for reference)
print('\n--- Common Imputation Strategies ---')
print('Numerical:  df[col].fillna(df[col].median())  # Median (robust to outliers)')
print('            df[col].fillna(df[col].mean())    # Mean')
print('Categorical: df[col].fillna(df[col].mode()[0]) # Mode')
print('Advanced:    from sklearn.impute import KNNImputer, IterativeImputer')
print('Drop:        df.dropna(subset=[col], thresh=0.7*len(df))')

## 3. Univariate Analysis — Distribution of Individual Features

In [ ]:
# Histograms for all numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    skew = df[col].skew()
    axes[i].text(0.95, 0.95, f'Skew: {skew:.2f}', transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distribution of All Numerical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots — Outlier visualization
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True, vert=True,
                          boxprops=dict(facecolor='lightblue', color='steelblue'),
                          medianprops=dict(color='red', linewidth=2))
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    axes[i].set_title(f'{col} (Outliers: {outliers})', fontsize=11)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Box Plots — Outlier Detection (IQR Method)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Bivariate Analysis — Relationships Between Features

In [ ]:
# Correlation matrix heatmap
corr = df.corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
plt.title('Correlation Matrix (Lower Triangle)', fontsize=14)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr['MedHouseVal'].drop('MedHouseVal').sort_values(key=abs, ascending=False)
print('\nCorrelation with Target (MedHouseVal):')
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 30)
    print(f'  {feat:15s} {val:+.3f} {bar}')

In [ ]:
# Scatter plots: Top features vs Target
top_features = target_corr.index[:4]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, feat in zip(axes, top_features):
    ax.scatter(df[feat], df['MedHouseVal'], alpha=0.1, s=5, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('MedHouseVal')
    r = df[feat].corr(df['MedHouseVal'])
    ax.set_title(f'{feat} vs Target (r={r:.3f})')
plt.suptitle('Top Features vs Target — Scatter Plots', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot (subset of most correlated features)
selected = list(top_features[:4]) + ['MedHouseVal']
sns.pairplot(df[selected].sample(2000, random_state=42), 
             diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10})
plt.suptitle('Pair Plot — Top Correlated Features', y=1.02, fontsize=14)
plt.show()

## 5. Outlier Detection — IQR & Z-Score Methods

In [ ]:
# Outlier summary for all columns
outlier_summary = []
for col in numerical_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    iqr_outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    z_outliers = (np.abs(stats.zscore(df[col].dropna())) > 3).sum()
    outlier_summary.append({
        'Feature': col,
        'IQR Outliers': iqr_outliers,
        'IQR %': round(iqr_outliers / len(df) * 100, 2),
        'Z-Score Outliers (|z|>3)': z_outliers,
        'Z-Score %': round(z_outliers / len(df) * 100, 2)
    })
    
outlier_df = pd.DataFrame(outlier_summary).sort_values('IQR %', ascending=False)
outlier_df.style.background_gradient(subset=['IQR %', 'Z-Score %'], cmap='Reds')

## 6. EDA Checklist — Reusable Template

Use this checklist for every new dataset:

| Step | Action | Code Snippet |
|------|--------|-------------|
| 1 | **Shape & Types** | `df.shape`, `df.dtypes`, `df.info()` |
| 2 | **First Look** | `df.head()`, `df.sample(5)`, `df.describe()` |
| 3 | **Missing Values** | `df.isnull().sum()`, `msno.matrix(df)` |
| 4 | **Duplicates** | `df.duplicated().sum()`, `df.drop_duplicates()` |
| 5 | **Distributions** | Histograms, `df[col].skew()`, KDE plots |
| 6 | **Outliers** | Box plots, IQR method, Z-score |
| 7 | **Correlations** | `df.corr()`, heatmap, pair plot |
| 8 | **Feature vs Target** | Scatter plots, groupby aggregation |
| 9 | **Categorical Encoding** | `value_counts()`, bar plots, crosstab |
| 10 | **Feature Engineering Ideas** | Log transforms, binning, interactions |